In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv('./data_csv/players_full_data_v2.csv')

df.head()

In [ ]:
df.shape

In [ ]:
df = df.drop(columns=['weekNumber', 'name', 'team_shortName'])

In [ ]:
df.shape

In [ ]:
df.isna().sum()

In [ ]:
# filtering out players that appear less than 10 times as we need both
# for training and testing
tmp = df['id'].tolist()
df = df[df['id'].apply(lambda x: tmp.count(x) >= 10)]

In [ ]:
df.shape

In [ ]:
# target
target_var = 'totalPoints'

# create X and y data frames
X = df.drop(columns=target_var)
y = df[target_var]

X.shape, y.shape

In [ ]:
from sklearn.model_selection import train_test_split

# Split test and train
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, stratify=X[['id']])

## RandomForest

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.compose import ColumnTransformer

# categorical columns
cat_columns = ['id', 'position', 'team_id', 'curr_match_opponent_id']

# define one hot encoder
ohe = OneHotEncoder(sparse_output=False)
ohe_transfromer = ColumnTransformer([('ohe', ohe, cat_columns)], remainder='passthrough')

# pipeline
pipe = Pipeline([
    ('ohe_transformer', ohe_transfromer),
    ('scaler', StandardScaler()),
    ('estimator', RandomForestClassifier())
])

pipe.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import mean_squared_error as MSE
import numpy as np

pred = pipe.predict(X_test)
np.sqrt(MSE(y_test, pred))

## XGBoost

In [ ]:
from xgboost import XGBRegressor, XGBClassifier
from sklearn.model_selection import cross_val_score

In [ ]:
estimator = XGBRegressor()

pipe = Pipeline([
    ('ohe_transformer', ohe_transfromer),
    ('scaler', StandardScaler()),
    ('estimator', estimator)
])

pipe.fit(X_train, y_train)

In [ ]:
pred = pipe.predict(X_test)


In [ ]:
np.sqrt(MSE(y_test, pred))

In [ ]:
import matplotlib.pyplot as plt

plt.hist(df['totalPoints'], bins=20)

In [ ]:
plt.hist(pred, bins=20)

In [ ]:
df_predictions = pd.read_csv('./data_csv/players_full_data_v2.csv')

In [ ]:
df_predictions[df_predictions['name'].str.contains('Lewan')]

In [ ]:
player = df.iloc[[4309]]
pipe.predict(player)

In [ ]:
X_test['predictions'] = pred

X_test.sort_values('predictions', ascending=False)

In [ ]:
df_predictions[df_predictions['id'] == 842]

In [ ]:
player